# 05 — CIC-IDS 2018 ReplicationReplicates the binary temporal leakage audit on a second, independent dataset (CIC-IDS 2018) to test whether the bias generalises beyond CIC-IDS 2017. Produces Table 6 of the paper.Temporal split: Week 1 (14–16 Feb) train → Week 2 (21–23 Feb) test. Random split: all captured days, 80/20 stratified.**Note:** this script includes a fix for a CIC-IDS 2018-specific data quality issue — several numeric flow features are stored as string type in the released CSVs and must be coerced to numeric *before*, not instead of, dropping genuinely non-numeric columns (Timestamp, IP addresses). See paper Section 7.1.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Download CIC-IDS 2018Downloads the nine CSV files used in this replication directly from the official CIC S3 bucket. Adjust `SAVE_2018` to your own Drive path.

In [2]:
import subprocess, os

SAVE_2018 = '/content/drive/MyDrive/New Approach/cicids2018/raw'

# Official CIC-IDS 2018 CSV files (processed by CICFlowMeter)
# Source: https://www.unb.ca/cic/datasets/ids-2018.html
# These are the Processed Traffic Data for ML Algorithms files
files = {
    'Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv',
    'Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv',
    'Friday-16-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv',
    'Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv',
    'Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv',
    'Friday-23-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv',
    'Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv',
    'Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv',
    'Friday-02-03-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv',
    'Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv':
        'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv',
}

for fname, url in files.items():
    dest = os.path.join(SAVE_2018, fname)
    if os.path.exists(dest):
        print(f'SKIP (exists): {fname}')
        continue
    print(f'Downloading: {fname} ...')
    result = subprocess.run(['wget', '-q', '-O', dest, url], capture_output=True)
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    if size > 10000:
        print(f'  OK  ({size/1e6:.1f} MB)')
    else:
        print(f'  FAILED or empty — check URL manually: {url}')
        os.remove(dest)

SKIP (exists): Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
SKIP (exists): Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv


## Step 2 — Main Audit Script

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import gc, warnings
warnings.filterwarnings('ignore')

RAW_2018  = '/content/drive/MyDrive/New Approach/cicids2018/raw'
SAVE_PATH = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed'
LABEL_COL    = 'Label'
RANDOM_STATE = 42
BENIGN_LABEL = 'Benign'

# ── FIX: added Timestamp and other known non-numeric columns ──
CATEGORICAL_EXCLUDE = ['Dst Port','Src Port',
                       'Destination Port','Source Port',
                       'Timestamp', 'Flow ID', 'Src IP', 'Dst IP',
                       'Source IP', 'Destination IP']

week_assign = {
    '14-02-2018': 'week1', '15-02-2018': 'week1', '16-02-2018': 'week1',
    '21-02-2018': 'week2', '22-02-2018': 'week2', '23-02-2018': 'week2',
    '28-02-2018': 'week3', '01-03-2018': 'week3', '02-03-2018': 'week3',
}

if 'df_all' not in dir() or len(df_all) == 0:
    print('Loading CIC-IDS 2018 (9 files)...\n')
    dfs = []
    for fname in sorted(os.listdir(RAW_2018)):
        if not fname.endswith('.csv'): continue
        week = None
        for date_key, wk in week_assign.items():
            if date_key in fname: week = wk; break
        if week is None: continue
        path = os.path.join(RAW_2018, fname)
        df = pd.read_csv(path, low_memory=False)
        df.columns = df.columns.str.strip()
        df = df[df[LABEL_COL] != LABEL_COL].reset_index(drop=True)
        df['week'] = week
        dfs.append(df)
        print(f'  {fname[:50]:50s} ({week}) | rows:{len(df):>8,}')
    df_all = pd.concat(dfs, ignore_index=True)
    del dfs; gc.collect()
    print(f'\nTotal: {len(df_all):,} rows')
else:
    print(f'Using existing df_all: {len(df_all):,} rows')


def preprocess_2018(df_tr, df_te, seed=RANDOM_STATE,
                    variance_thresh=0.01, corr_thresh=0.95):
    try:
        X_tr     = df_tr.drop(columns=[LABEL_COL,'week'], errors='ignore').copy()
        X_te     = df_te.drop(columns=[LABEL_COL,'week'], errors='ignore').copy()
        y_tr_raw = df_tr[LABEL_COL].copy()
        y_te_raw = df_te[LABEL_COL].copy()

        y_tr_raw = y_tr_raw.apply(lambda x: 'BENIGN' if str(x).strip()==BENIGN_LABEL else 'ATTACK')
        y_te_raw = y_te_raw.apply(lambda x: 'BENIGN' if str(x).strip()==BENIGN_LABEL else 'ATTACK')

        # ── STEP 1: drop KNOWN non-numeric identifier/text columns first ──
        # (Timestamp, IPs, Flow ID — these must be dropped BEFORE coercion
        # since they are genuinely textual, not just string-typed numbers)
        for X in [X_tr, X_te]:
            drop_cols = [c for c in X.columns if c in CATEGORICAL_EXCLUDE]
            X.drop(columns=drop_cols, inplace=True, errors='ignore')

        # ── STEP 2: coerce EVERYTHING ELSE to numeric first ──
        # (this correctly handles columns that are string-typed numbers,
        # e.g. Flow Duration stored as '112640768' instead of 112640768)
        X_tr = X_tr.apply(pd.to_numeric, errors='coerce')
        X_te = X_te.apply(pd.to_numeric, errors='coerce')

        # ── STEP 3: only NOW drop columns that are still unusable ──
        # (fully NaN after coercion = genuinely non-numeric junk,
        # not just string-typed numbers)
        all_nan_tr = X_tr.columns[X_tr.isna().all()].tolist()
        all_nan_te = X_te.columns[X_te.isna().all()].tolist()
        if all_nan_tr:
            print(f'  Dropping genuinely non-numeric columns (train): {all_nan_tr}')
        X_tr.drop(columns=all_nan_tr, inplace=True)
        X_te.drop(columns=[c for c in all_nan_te if c in X_te.columns],
                 inplace=True, errors='ignore')

        X_tr.replace([np.inf,-np.inf], np.nan, inplace=True)
        X_te.replace([np.inf,-np.inf], np.nan, inplace=True)

        mask_tr = X_tr.notna().all(axis=1)
        mask_te = X_te.notna().all(axis=1)
        print(f'  Train rows kept after NaN-drop: {mask_tr.sum():,}/{len(mask_tr):,}')
        print(f'  Test rows kept after NaN-drop:  {mask_te.sum():,}/{len(mask_te):,}')

        X_tr = X_tr[mask_tr].reset_index(drop=True)
        y_tr_raw = y_tr_raw[mask_tr].reset_index(drop=True)
        X_te = X_te[mask_te].reset_index(drop=True)
        y_te_raw = y_te_raw[mask_te].reset_index(drop=True)

        common = [c for c in X_tr.columns if c in X_te.columns]
        print(f'  Common numeric features: {len(common)}')
        if len(common) == 0 or mask_te.sum() == 0:
            print('  [ERROR] no usable rows/columns remain')
            return (None,)*5

        X_tr = X_tr[common].copy(); X_te = X_te[common].copy()

        vt   = VarianceThreshold(threshold=variance_thresh)
        arr  = vt.fit_transform(X_tr)
        cols = np.array(common)[vt.get_support()]
        X_tr = pd.DataFrame(arr, columns=cols)
        X_te = pd.DataFrame(vt.transform(X_te), columns=cols)

        corr  = X_tr.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
        drop  = [c for c in upper.columns if any(upper[c]>=corr_thresh)]
        X_tr.drop(columns=drop, inplace=True)
        X_te.drop(columns=drop, inplace=True, errors='ignore')
        feature_names = X_tr.columns.tolist()
        print(f'  Final features: {len(feature_names)}')

        scaler  = StandardScaler()
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_names)
        X_te_sc = pd.DataFrame(scaler.transform(X_te),     columns=feature_names)

        le = LabelEncoder()
        le.fit(['BENIGN','ATTACK'])
        y_tr = pd.Series(le.transform(y_tr_raw), name='label')
        y_te = pd.Series(le.transform(y_te_raw), name='label')

        print(f'  Test label distribution: {dict(Counter(y_te_raw))}')

        counts  = Counter(y_tr)
        min_cls = min(counts, key=counts.get)
        min_cnt = counts[min_cls]
        maj_cnt = counts[max(counts, key=counts.get)]
        k       = max(1, min(5, min_cnt-1))
        target  = max(min_cnt, min(30_000, maj_cnt))
        strat   = {min_cls: target}
        print(f'  SMOTE k={k} | {min_cnt}\u2192{target} | train:{dict(Counter(y_tr_raw))}')

        smote = SMOTE(random_state=seed, k_neighbors=k, sampling_strategy=strat)
        X_sm, y_sm = smote.fit_resample(X_tr_sc, y_tr)
        print(f'  After SMOTE:{len(y_sm):,} | Test:{len(y_te):,}')
        return X_sm, y_sm, X_te_sc, y_te, le

    except Exception as e:
        import traceback; traceback.print_exc()
        return (None,)*5


def evaluate(model, X_te, y_te, le, model_name, split_name):
    preds = model.predict(X_te)
    classes    = list(le.classes_)
    benign_idx = classes.index('BENIGN')
    attack_idx = classes.index('ATTACK')
    acc = accuracy_score(y_te, preds)
    f1m = f1_score(y_te, preds, average='macro', zero_division=0)
    f1w = f1_score(y_te, preds, average='weighted', zero_division=0)
    atk_mask   = (y_te == attack_idx)
    atk_recall = (preds[atk_mask]==attack_idx).mean() if atk_mask.any() else 0.0
    cm  = confusion_matrix(y_te, preds, labels=[benign_idx, attack_idx])
    TN, FP = cm[0,0], cm[0,1]
    FPR = FP/(FP+TN) if (FP+TN)>0 else 0.0
    try:
        proba = model.predict_proba(X_te)[:,attack_idx]
        roc   = roc_auc_score((y_te==attack_idx).astype(int), proba)
    except Exception:
        roc = float('nan')
    print(f'  [{split_name:8s}] {model_name:14s} | Acc:{acc:.4f} | F1m:{f1m:.4f} | '
          f'Recall:{atk_recall:.4f} | FPR:{FPR:.4f} | AUC:{roc:.4f}')
    return {'model':model_name,'split':split_name,
            'accuracy':round(acc,4),'f1_macro':round(f1m,4),
            'f1_weighted':round(f1w,4),'attack_recall':round(atk_recall,4),
            'FPR':round(FPR,4),'ROC_AUC':round(roc,4) if not np.isnan(roc) else None}


def cap_df(df, n=200_000, seed=RANDOM_STATE):
    if len(df) <= n: return df.copy()
    return (df.groupby(LABEL_COL, group_keys=False)
              .apply(lambda x: x.sample(min(len(x), max(6,int(n*len(x)/len(df)))),
                                        random_state=seed))
              .reset_index(drop=True))

print('\n' + '='*65)
print('TEMPORAL SPLIT — Week1 train / Week2 test')
print('='*65)
df_train_t = df_all[df_all['week']=='week1'].copy()
df_test_t  = df_all[df_all['week']=='week2'].copy()

train_atk = set(df_train_t[LABEL_COL].unique()) - {BENIGN_LABEL}
test_atk  = set(df_test_t[LABEL_COL].unique())  - {BENIGN_LABEL}
unseen    = test_atk - train_atk
n_unseen  = int((df_test_t[LABEL_COL].isin(unseen)).sum())
n_total_te = len(df_test_t)
print(f'Unseen attack classes in week2: {unseen}')
print(f'Unseen rows: {n_unseen:,} ({n_unseen/n_total_te*100:.1f}% of week2)')

df_train_t = cap_df(df_train_t, n=200_000)
df_test_t  = cap_df(df_test_t,  n=100_000)

result = preprocess_2018(df_train_t, df_test_t)
assert result[0] is not None, "Temporal split failed"
X_tr_temp,y_tr_temp,X_te_temp,y_te_temp,le_temp = result
del df_train_t, df_test_t; gc.collect()
print('Temporal split ready \u2713')

print('\n' + '='*65)
print('TEMPORAL SPLIT — Week1 train / Week2 test')
print('='*65)
df_train_t = df_all[df_all['week']=='week1'].copy()
df_test_t  = df_all[df_all['week']=='week2'].copy()

train_atk = set(df_train_t[LABEL_COL].unique()) - {BENIGN_LABEL}
test_atk  = set(df_test_t[LABEL_COL].unique())  - {BENIGN_LABEL}
unseen    = test_atk - train_atk
n_unseen  = int((df_test_t[LABEL_COL].isin(unseen)).sum())
n_total_te = len(df_test_t)
print(f'Unseen attack classes in week2: {unseen}')
print(f'Unseen rows: {n_unseen:,} ({n_unseen/n_total_te*100:.1f}% of week2)')

df_train_t = cap_df(df_train_t, n=200_000)
df_test_t  = cap_df(df_test_t,  n=100_000)

result = preprocess_2018(df_train_t, df_test_t)
assert result[0] is not None, "Temporal split failed"
X_tr_temp,y_tr_temp,X_te_temp,y_te_temp,le_temp = result
del df_train_t, df_test_t; gc.collect()
print('Temporal split ready ✓')

print('\n' + '='*65)
print('RANDOM SPLIT — all weeks combined, 80/20')
print('='*65)
df_rand = cap_df(df_all, n=200_000)
X_all_r = df_rand.drop(columns=[LABEL_COL,'week'], errors='ignore')
y_all_r = df_rand[LABEL_COL]
X_r_tr,X_r_te,y_r_tr,y_r_te = train_test_split(
    X_all_r, y_all_r, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all_r)
df_r_tr = pd.concat([X_r_tr, y_r_tr], axis=1)
df_r_te = pd.concat([X_r_te, y_r_te], axis=1)

result = preprocess_2018(df_r_tr, df_r_te)
assert result[0] is not None, "Random split failed"
X_tr_rand,y_tr_rand,X_te_rand,y_te_rand,le_rand = result
del df_rand,df_r_tr,df_r_te,X_all_r,y_all_r; gc.collect()
print('Random split ready ✓')

results = []
print('\n' + '='*70)
print('CIC-IDS 2018 — TEMPORAL LEAKAGE AUDIT (FIXED)')
print('='*70)
for model_name, model in models.items():
    print(f'\n--- {model_name} ---')
    model.fit(X_tr_rand, y_tr_rand)
    results.append(evaluate(model, X_te_rand, y_te_rand, le_rand, model_name, 'Random'))
    gc.collect()
    model.fit(X_tr_temp, y_tr_temp)
    results.append(evaluate(model, X_te_temp, y_te_temp, le_temp, model_name, 'Temporal'))
    gc.collect()

df_r = pd.DataFrame(results)
print('\n\n' + '='*70)
print('RESULTS — CIC-IDS 2018 TEMPORAL LEAKAGE AUDIT (FIXED)')
print('='*70)
print(f'\n{"Model":14s} {"Split":10s} {"Accuracy":>10} {"F1-macro":>10} {"Atk-Recall":>12} {"FPR":>8}')
for mn in models.keys():
    for sp in ['Random','Temporal']:
        row = df_r[(df_r.model==mn)&(df_r.split==sp)].iloc[0]
        print(f'{mn:14s} {sp:10s} {row.accuracy:>10.4f} {row.f1_macro:>10.4f} '
              f'{row.attack_recall:>12.4f} {row.FPR:>8.4f}')
    print()

print('INFLATION (Random − Temporal):')
for mn in models.keys():
    rand = df_r[(df_r.model==mn)&(df_r.split=='Random')].iloc[0]
    temp = df_r[(df_r.model==mn)&(df_r.split=='Temporal')].iloc[0]
    print(f'{mn:14s} ΔAcc:{rand.accuracy-temp.accuracy:+.4f} '
          f'ΔF1m:{rand.f1_macro-temp.f1_macro:+.4f} '
          f'ΔRecall:{rand.attack_recall-temp.attack_recall:+.4f}')

df_r.to_csv(SAVE_PATH + '/cicids2018_audit.csv', index=False)
print(f'\nSaved → cicids2018_audit.csv')

Loading CIC-IDS 2018 (9 files)...

  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv    (week3) | rows:1,048,575
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv    (week1) | rows:1,048,574
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv    (week2) | rows:1,048,575
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv  (week3) | rows: 331,100
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv  (week1) | rows:1,048,575
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv  (week2) | rows:1,048,575
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv (week1) | rows:1,048,575
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv (week2) | rows:1,048,575
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv (week3) | rows: 613,071

Total: 8,284,195 rows

TEMPORAL SPLIT — Week1 train / Week2 test
Unseen attack classes in week2: {'Brute Force -Web', 'Brute Force -XSS', 'DDOS attack-LOIC-UDP', 'DDOS attack-HOIC', 'SQL Injection'}
Unseen rows: 688,670 (21.9% of week2)
  Train rows kept after NaN

NameError: name 'models' is not defined

## Step 3 — Feature Completeness Verification (DDoS-HOIC)Confirms the zero-recall result on DDoS-HOIC traffic under temporal evaluation is a genuine distributional-shift finding, not an artefact of missing or corrupted features: all 686,012 DDoS-HOIC flows in the Week 2 test set retain complete, finite values across every feature after preprocessing (paper Section 4.5 / 7.2).

In [4]:
import pandas as pd
import numpy as np

# ── rebuild the DDoS-HOIC test rows exactly as used in the temporal audit ──
# (assumes df_all is still in memory from your CIC-IDS 2018 session;
# if not, reload from the pickle/CSV checkpoint first)

print("Checking feature completeness for DDoS-HOIC rows specifically...")

df_test_check = df_all[df_all['week']=='week2'].copy()
hoic_rows = df_test_check[df_test_check[LABEL_COL] == 'DDOS attack-HOIC'].copy()
print(f"Raw DDoS-HOIC rows in week2: {len(hoic_rows):,}")

# apply the SAME cleaning steps as preprocess_2018, but track HOIC specifically
X_hoic = hoic_rows.drop(columns=[LABEL_COL,'week'], errors='ignore').copy()

CATEGORICAL_EXCLUDE = ['Dst Port','Src Port','Destination Port','Source Port',
                       'Timestamp', 'Flow ID', 'Src IP', 'Dst IP',
                       'Source IP', 'Destination IP']
for col in CATEGORICAL_EXCLUDE:
    if col in X_hoic.columns:
        X_hoic.drop(columns=[col], inplace=True)

# coerce to numeric — same as the fixed pipeline
X_hoic_numeric = X_hoic.apply(pd.to_numeric, errors='coerce')

print(f"\nColumns that are ENTIRELY NaN for HOIC specifically:")
all_nan_cols = X_hoic_numeric.columns[X_hoic_numeric.isna().all()].tolist()
print(f"  {all_nan_cols if all_nan_cols else 'NONE — no fully-NaN columns'}")

print(f"\nPer-column NaN percentage for HOIC rows (top 15 worst):")
nan_pct = (X_hoic_numeric.isna().sum() / len(X_hoic_numeric) * 100).sort_values(ascending=False)
print(nan_pct.head(15))

# inf check
inf_check = np.isinf(X_hoic_numeric.select_dtypes(include=[np.number])).sum()
print(f"\nColumns with inf values (top 10):")
print(inf_check[inf_check>0].sort_values(ascending=False).head(10))

# row-level completeness — what fraction of HOIC rows would SURVIVE
# the full row-wise dropna used in the actual pipeline?
X_hoic_clean = X_hoic_numeric.replace([np.inf,-np.inf], np.nan)
row_complete_mask = X_hoic_clean.notna().all(axis=1)
print(f"\nHOIC rows with ALL features present (would survive dropna): "
      f"{row_complete_mask.sum():,} / {len(row_complete_mask):,} "
      f"({row_complete_mask.mean()*100:.1f}%)")

# compare feature-completeness rate to BENIGN and to other week2 attacks
print(f"\n--- Comparison: feature completeness rate by class in week2 ---")
for cls in df_test_check[LABEL_COL].unique():
    sub = df_test_check[df_test_check[LABEL_COL]==cls].drop(
        columns=[LABEL_COL,'week'], errors='ignore')
    for col in CATEGORICAL_EXCLUDE:
        if col in sub.columns:
            sub = sub.drop(columns=[col])
    sub_numeric = sub.apply(pd.to_numeric, errors='coerce').replace(
        [np.inf,-np.inf], np.nan)
    complete_rate = sub_numeric.notna().all(axis=1).mean() * 100
    print(f"  {cls:30s}: {complete_rate:.1f}% rows fully complete "
          f"(n={len(sub):,})")

# ── the critical comparison: HOIC's actual feature VALUES for the
# columns that survived preprocessing — are they populated with
# real, non-trivial numbers, or degenerate placeholders (e.g. all 0)? ──
print(f"\n--- HOIC surviving rows: are feature values non-trivial? ---")
X_hoic_survived = X_hoic_clean[row_complete_mask]
print(f"Rows analyzed: {len(X_hoic_survived):,}")
print(f"\nMean, std for first 10 features (HOIC survivors):")
print(X_hoic_survived.iloc[:,:10].agg(['mean','std']).T)

zero_var_hoic = X_hoic_survived.columns[X_hoic_survived.std() < 1e-9].tolist()
print(f"\nHOIC-survivor columns with ZERO variance (all identical value): "
      f"{len(zero_var_hoic)}")
if zero_var_hoic:
    print(f"  {zero_var_hoic}")
    for col in zero_var_hoic[:5]:
        print(f"    {col} = constant at {X_hoic_survived[col].iloc[0]}")

Checking feature completeness for DDoS-HOIC rows specifically...
Raw DDoS-HOIC rows in week2: 686,012

Columns that are ENTIRELY NaN for HOIC specifically:
  NONE — no fully-NaN columns

Per-column NaN percentage for HOIC rows (top 15 worst):
Protocol            0.0
Flow Duration       0.0
Tot Fwd Pkts        0.0
Tot Bwd Pkts        0.0
TotLen Fwd Pkts     0.0
TotLen Bwd Pkts     0.0
Fwd Pkt Len Max     0.0
Fwd Pkt Len Min     0.0
Fwd Pkt Len Mean    0.0
Fwd Pkt Len Std     0.0
Bwd Pkt Len Max     0.0
Bwd Pkt Len Min     0.0
Bwd Pkt Len Mean    0.0
Bwd Pkt Len Std     0.0
Flow Byts/s         0.0
dtype: float64

Columns with inf values (top 10):
Series([], dtype: int64)

HOIC rows with ALL features present (would survive dropna): 686,012 / 686,012 (100.0%)

--- Comparison: feature completeness rate by class in week2 ---
  Benign                        : 99.5% rows fully complete (n=2,457,055)
  Brute Force -Web              : 100.0% rows fully complete (n=611)
  Brute Force -XSS        